<a href="https://colab.research.google.com/github/mdvdmitrov-eng/project/blob/main/%D0%BA%D0%BE%D0%B4_%D1%82%D0%B5%D1%81%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# test_quote_generator.py

import pytest
from unittest.mock  import patch
import requests


def get_quote():
   try:
        response = requests.get("https://api.forismatic.com/api/1.0/", timeout=5)
        response.raise_for_status()
        data = response.json()
        text = data.get("quoteText", "").strip()
        author = data.get("quoteAuthor", "").strip()
        if not author:
            author = "Неизвестный"
        return f"Цитата: {text}\nАвтор: {author}"
   except (requests.ConnectionError, requests.HTTPError, requests.Timeout) as e:
        return f"Ошибка: {str(e)}"

# 1...Тест  получения цитаты с указанием автора
def test_get_quote():
    mock_response_data = {
        "quoteText": "Тестирование — это важный этап.",
        "quoteAuthor": "Программист"
    }

    with patch("requests.get") as mock_get:
        mock_get.return_value.status_code = 200
        mock_get.return_value.json.return_value = mock_response_data

        result = get_quote()
        assert result == "Цитата: Тестирование — это важный этап.\nАвтор: Программист"

# 2... Тест, когда не указан автор
def test_get_quote_unknown_author():
    mock_response_data = {
        "quoteText": "Код без тестов — это неудобно.",
        "quoteAuthor": ""
    }
    with patch("requests.get") as mock_get:
        mock_get.return_value.status_code = 200
        mock_get.return_value.json.return_value = mock_response_data

        result = get_quote()
        assert result == "Цитата: Код без тестов — это неудобно.\nАвтор: Неизвестный"

# 3... Тест ошибки сети
def test_get_quote_network_error():
    with patch("requests.get", side_effect=requests.ConnectionError("Нет связи")):
        result = get_quote()
        assert result.startswith("Ошибка:")

# 4... Тест ошибки HTTP (403)
def test_get_quote_http_error():
    with patch("requests.get") as mock_get:
        mock_get.return_value.raise_for_status.side_effect = requests.HTTPError("403 Forbidden")

        result = get_quote()
        assert result.startswith("Ошибка:")

# 5... Тест таймаута
def test_get_quote_timeout():
    with patch("requests.get", side_effect=requests.Timeout("Таймаут")):
        result = get_quote()
        assert result.startswith("Ошибка:")

test_get_quote()
test_get_quote_unknown_author()
test_get_quote_network_error()
test_get_quote_http_error()
test_get_quote_timeout()